In [ ]:
import pandas as pd
import numpy as np
import glob

from sklearn import metrics
from functools import partial
from scipy.optimize import fmin

### plan
- need to load the OOF prediction data for each base model under consideration
- combine them into a single dataframe where each column has a different model's predictions
- consider roc_auc_score(y_true, weighted_average_prediction) as the loss function that we need to optimize
    - optimize the weight coefs applied to each model (weighted average prediction is the weighted linear combination of models' preds)


- once optimized coefs are found, apply those to the base model predictions to come up with our TRUE FINAL predictions

In [ ]:
class OptimizeAUC:
    def __init__(self):
        self.coef_ = 0

    def _auc(self, coef, X, y):
        x_coef = X * coef
        predictions = np.sum(x_coef, axis=1)
        auc_score = metrics.roc_auc_score(y, predictions)
        return -1.0 * auc_score
    
    def fit(self, X, y):
        partial_loss = partial(self._auc, X=X, y=y)    
        init_coef = np.random.dirichlet(np.ones(shape=(X.shape[1])))
        self.coef_ = fmin(partial_loss, init_coef, disp=True)

    def predict(self, X):
        X_optim_weights = X * self.coef_
        predictions = np.sum(X_optim_weights, axis=1)
        return predictions

In [ ]:
def run_training(pred_df, fold):
    train_df = pred_df[pred_df.kfold != fold].reset_index(drop=True)
    valid_df = pred_df[pred_df.kfold == fold].reset_index(drop=True)

    xtrain = train_df[["lr_pred", "lr_cnt_pred", "rf_svd_pred"]].values
    xvalid = valid_df[["lr_pred", "lr_cnt_pred", "rf_svd_pred"]].values

    opt = OptimizeAUC()
    opt.fit(xtrain, train_df.sentiment.values)
    preds = opt.predict(xvalid)
    auc_score = metrics.roc_auc_score(valid_df.sentiment.values, preds)
    print(f"fold={fold}, AUC={auc_score}")
    return opt.coef_

In [ ]:
files = glob.glob("./model_oof_preds/*.csv")

df = None
for f in files:
    if df is None:
        df = pd.read_csv(f)
    else:
        temp_df = pd.read_csv(f)
        df = df.merge(temp_df, on='id', how='left')

targets = df.sentiment.values
pred_cols = ["lr_pred", "lr_cnt_pred", "rf_svd_pred"]
    
oof_coefs = []

for f in range(5):
    oof_coefs.append(run_training(df, f))

coefs = np.array(oof_coefs)
avg_coefs = np.mean(coefs, axis=0)
print(f"average OOF coef values: {avg_coefs}")

opt_weights_avg = (avg_coefs[0] * df.lr_pred.values + avg_coefs[1] * df.lr_cnt_pred.values + avg_coefs[2] * df.rf_svd_pred.values)
print("optimized coefficient weights AUC:")
print(metrics.roc_auc_score(targets, opt_weights_avg))